In [ ]:
import os
import sys

for f in os.listdir():
    if f.startswith("SolveSel") or f.endswith(".tokens") or f.endswith(".interp") or (f.endswith(".py") and f != "tu_notebook.py"):
        try:
            os.remove(f)
        except:
            pass

In [ ]:
!pip install antlr4-python3-runtime==4.13.2
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.13.2 which is incompatible.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2089k  100 2089k    0     0  4020k      0 --:--:-- --:--:-- --:--:-- 4019k


In [ ]:
%%writefile SolveSel.g4
grammar SolveSel;

root: (action)* EOF;

action : expr '=' expr ';' #Ecuacion
       | 'solve' '(' ID ')' ';' #Resolver
       | 'print' expr ';' #Mostrar
       ;

expr : expr op=('*'|'/') expr #MulDiv
     | expr op=('+'|'-') expr #AddSub
     | term #Atom
     | '(' expr ')' #Parens
     ;

term : NUMBER ID #CoeffVar
     | ID #SoloVar
     | NUMBER #SoloNum
     ;

ID     : [a-zA-Z_][a-zA-Z0-9_]* ;
NUMBER : [0-9]+ ('.' [0-9]+)? ;
WS     : [ \t\r\n]+ -> skip ;

Writing SolveSel.g4


In [ ]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 SolveSel.g4 -visitor -no-listener -o .

In [ ]:
import sys
import importlib
from antlr4 import *
for m in ['SolveSelLexer', 'SolveSelParser', 'SolveSelVisitor']:
    if m in sys.modules:
        importlib.reload(sys.modules[m])

from SolveSelLexer import SolveSelLexer
from SolveSelParser import SolveSelParser
from SolveSelVisitor import SolveSelVisitor

class EvalVisitor(SolveSelVisitor):
    def visitEcuacion(self, ctx):
        if ctx.expr(0) and ctx.expr(1):
            print(f"[*] Ecuación reconocida: {ctx.expr(0).getText()} = {ctx.expr(1).getText()}")
        return self.visitChildren(ctx)

    def visitCoeffVar(self, ctx):
        print(f"    -> Término Mixto: {ctx.NUMBER().getText()} con variable {ctx.ID().getText()}")
        return self.visitChildren(ctx)

    def visitSoloVar(self, ctx):
        print(f"    -> Variable sola: {ctx.ID().getText()}")
        return self.visitChildren(ctx)

    def visitResolver(self, ctx):
        print(f"[!] ORDEN: Resolver sistema '{ctx.ID().getText()}'")

input_code = """
2x + 3y = 10;
1x - 1y = 5;

5a + 2b = 20;
3m - 4n = 12;

7p + q = 9;
9r - 2s = 15;

solve(sistema1);
"""

lexer = SolveSelLexer(InputStream(input_code))
stream = CommonTokenStream(lexer)
parser = SolveSelParser(stream)
tree = parser.root()

if parser.getNumberOfSyntaxErrors() == 0:
    visitor = EvalVisitor()
    visitor.visit(tree)

[*] Ecuación reconocida: 2x+3y = 10
    -> Término Mixto: 2 con variable x
    -> Término Mixto: 3 con variable y
[*] Ecuación reconocida: 1x-1y = 5
    -> Término Mixto: 1 con variable x
    -> Término Mixto: 1 con variable y
[*] Ecuación reconocida: 5a+2b = 20
    -> Término Mixto: 5 con variable a
    -> Término Mixto: 2 con variable b
[*] Ecuación reconocida: 3m-4n = 12
    -> Término Mixto: 3 con variable m
    -> Término Mixto: 4 con variable n
[*] Ecuación reconocida: 7p+q = 9
    -> Término Mixto: 7 con variable p
    -> Variable sola: q
[*] Ecuación reconocida: 9r-2s = 15
    -> Término Mixto: 9 con variable r
    -> Término Mixto: 2 con variable s
[!] ORDEN: Resolver sistema 'sistema1'
